In [ ]:
import pandas as pd
import difflib
import os

DOCS_DIR = "../docs/"
INTER_DIR = "../data/02_intermediate/"

print("Starte hybrides Identity Resolution (Heuristiken + Fuzzy Matching)...")

# Justiz-Kategorien als Referenz laden
df_jus = pd.read_csv(os.path.join(INTER_DIR, 'justiz_mapped.csv'), sep=';')
jus_kategorien = df_jus['straftat_bezeichnung'].dropna().unique().tolist()

# Leere Mapping-Tabelle laden (aus Notebook 04 erstellt)
mapping_path = os.path.join(DOCS_DIR, 'mapping_table.csv')
df_mapping = pd.read_csv(mapping_path, sep=';')

def regelbasiertes_mapping(pks_text, ziel_kategorien):
    """Hybrid-Ansatz: Erst regelbasiert per Schlüsselwort, dann Fuzzy Matching als Fallback."""
    text_lower = str(pks_text).lower()
    
    # Regelbasierte Zuordnung über juristische Schlüsselwörter
    if any(wort in text_lower for wort in ["mord", "totschlag", "tötung"]):
        return "Mord und Totschlag"
    if any(wort in text_lower for wort in ["diebstahl", "einbruch", "kiosk", "hehlerei", "unterschlagung"]):
        return "Diebstahl und Unterschlagung"
    if any(wort in text_lower for wort in ["betrug", "untreue", "erschleichen"]):
        return "Betrug"
    if any(wort in text_lower for wort in ["btmg", "betäubungsmittel", "kokain", "heroin", "cannabis", "lsd", "amphetamin", "drogen", "nps"]):
        return "Straftaten nach dem Betäubungsmittelgesetz"
    if any(wort in text_lower for wort in ["raub", "erpressung", "menschenraub", "geiselnahme"]):
        return "Raub u. Erpressung,räuber. Angriff auf Kraftfahrer"
    if any(wort in text_lower for wort in ["brandstiftung", "explosion", "umwelt", "gewässer", "abfall"]):
        return "Gemeingefährliche einschl. Umweltstraftaten (o.V.)"
    if "vergewaltigung" in text_lower or "sexuell" in text_lower or "prostituierten" in text_lower or "exhibitionistisch" in text_lower:
        return "Vergewaltigung" 
    if any(wort in text_lower for wort in ["verkehr", "straßenverkehr"]):
        return "Straftaten im Straßenverkehr nach dem StGB"
    if "aufenthalt" in text_lower or "asyl" in text_lower or "einschleusen" in text_lower:
        return "Straftaten nach dem Aufenthaltsgesetz"
    if "gefährliche" in text_lower or "schwere körperverletzung" in text_lower or "verstümmelung" in text_lower:
        return "Gefährliche und schwere Körperverletzung"
    if "körperverletzung" in text_lower:
        return "Körperverletzung"
    
    # Fuzzy Matching (difflib) als Fallback für nicht abgedeckte Delikte
    # Cutoff bei 0.5, um False Positives zu vermeiden
    treffer = difflib.get_close_matches(str(pks_text), ziel_kategorien, n=1, cutoff=0.5)
    if treffer:
        return treffer[0]
        
    return "NICHT GEFUNDEN - BITTE MANUELL EINTRAGEN"

# Mapping auf alle PKS-Delikte anwenden
print("Wende Heuristiken und Fuzzy Matching an...")
df_mapping['justiz_straftat_bezeichnung_mapped'] = df_mapping['pks_straftat_bezeichnung'].apply(
    lambda x: regelbasiertes_mapping(x, jus_kategorien)
)

output_path = os.path.join(DOCS_DIR, 'mapping_table_autofilled_v2.csv')
df_mapping.to_csv(output_path, index=False, sep=';')

# Auswertung
missing = df_mapping[df_mapping['justiz_straftat_bezeichnung_mapped'] == "NICHT GEFUNDEN - BITTE MANUELL EINTRAGEN"]
print(f"\nMapping abgeschlossen: {len(df_mapping)} Einträge, davon {len(missing)} noch manuell zu prüfen.")
print(f"Ergebnis gespeichert: {output_path}")
display(df_mapping.sample(10))